### Imports

In [10]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor

### Import Cleaned CSV

In [11]:
final_df = pd.read_csv('./new_dataset/cleaned_data_unscaled.csv')

y = final_df["productivity_score"]
X = final_df.drop(columns = ["productivity_score"])

### Split Data (Train and Test)

In [ ]:
# split dataset into training and test sets (random_state ensures reproducible results)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42
)

print("Training set:", X_trainval.shape)
print("Test set:", X_test.shape)

Training set: (3600, 22)
Test set: (900, 22)


### Scale Data

In [13]:
# standardize numerical features to mean=0 and std=1
scaler = StandardScaler()

cols_to_exclude = ["job_role_Designer", "job_role_Developer", "job_role_Manager", "job_role_Marketer", "job_role_Writer", "deadline_pressure_level_Low", "deadline_pressure_level_Medium", "deadline_pressure_level_High"]
cols_to_scale = X_trainval.columns.difference(cols_to_exclude)

# models go HERE!!!!
models = {
    "linear_regression": LinearRegression(),
    "ridge_l2": Ridge(alpha = 1.0),
    "decision_tree": DecisionTreeRegressor(random_state = 123),
    "xgboost": XGBRegressor(
        n_estimators = 500,
        learning_rate = 0.05,
        max_depth = 6,
        random_state = 42
    )
}

k = 5
kf = KFold(n_splits = k, shuffle = True, random_state = 42)

def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

results = {name: {"mae": [], "rmse": [], "r2": []} for name in models}

# loop through our folds and fit the data
for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval), start = 1):
    X_train = X_trainval.iloc[train_idx].copy()
    y_train = y_trainval.iloc[train_idx].copy()
    X_val = X_trainval.iloc[val_idx].copy()
    y_val = y_trainval.iloc[val_idx].copy()

    # fitting scalar to train data only
    scaler = StandardScaler()
    X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
    X_val[cols_to_scale] = scaler.transform(X_val[cols_to_scale])

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

        mae, rmse, r2 = metrics(y_val, preds)
        results[name]["mae"].append(mae)
        results[name]["rmse"].append(rmse)
        results[name]["r2"].append(r2)

# printing da results
for name in models:
    mae_mean = np.mean(results[name]["mae"])
    mae_std  = np.std(results[name]["mae"])

    rmse_mean = np.mean(results[name]["rmse"])
    rmse_std  = np.std(results[name]["rmse"])

    r2_mean = np.mean(results[name]["r2"])
    r2_std  = np.std(results[name]["r2"])

    print(f"\n{name}")
    print(f"MAE : {mae_mean:.4f} ± {mae_std:.4f}")
    print(f"RMSE: {rmse_mean:.4f} ± {rmse_std:.4f}")
    print(f"R^2 : {r2_mean:.4f} ± {r2_std:.4f}")



linear_regression
MAE : 4.0153 ± 0.0769
RMSE: 5.0267 ± 0.1508
R^2 : 0.8769 ± 0.0065

ridge_l2
MAE : 4.0150 ± 0.0763
RMSE: 5.0265 ± 0.1504
R^2 : 0.8769 ± 0.0065

decision_tree
MAE : 6.8662 ± 0.2558
RMSE: 8.6452 ± 0.2961
R^2 : 0.6362 ± 0.0143

xgboost
MAE : 4.4423 ± 0.1092
RMSE: 5.5954 ± 0.1528
R^2 : 0.8475 ± 0.0075


### Final Evaluation

In [14]:
best_model_name = "ridge_l2"
best_model = models[best_model_name]

final_scaler = StandardScaler()
X_trainval_scaled = X_trainval.copy()
X_test_scaled = X_test.copy()

X_trainval_scaled[cols_to_scale] = final_scaler.fit_transform(X_trainval_scaled[cols_to_scale])
X_test_scaled[cols_to_scale] = final_scaler.transform(X_test_scaled[cols_to_scale])

best_model.fit(X_trainval_scaled, y_trainval)
test_preds = best_model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, test_preds)
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
r2 = r2_score(y_test, test_preds)

print(f"Final Test on: {best_model_name}")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2 : {r2:.4f}")

Final Test on: ridge_l2
MAE : 3.9424
RMSE: 4.9099
R^2 : 0.8788


### Baseline (Mean Predictor)

In [15]:
# Baseline predicting the training mean for the test set
y_pred_baseline = np.full(len(y_test), y_trainval.mean())

baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
baseline_r2 = r2_score(y_test, y_pred_baseline)

print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Baseline R^2: {baseline_r2:.4f}")

Baseline MAE: 11.25
Baseline RMSE: 14.11
Baseline R^2: -0.0004
